# Árvore de Cenários para Determinação de Propriedades de Risco para Febre Aftosa
## Convênio nº 24/2024 - IMA (Instituto Mineiro de Agropecuária)

## Importação das bibliotecas

In [ ]:
import os
import pandas as pd
from pathlib import Path
import re
from scipy.spatial.distance import cdist

## Diretório dos dados

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)# Inspecionar colunas

In [ ]:
from src.data_io import (
    RAW_DIR,
    load_propriedades_processed,
    load_coordenadorias_dict,
    build_coordenadoria_mappings,
    read_coordenadoria_csv,
)

propriedades_df = load_propriedades_processed()

In [ ]:
print(propriedades_df.columns)

In [ ]:
total_propriedades = len(propriedades_df)
total_propriedades

### Detalhamento de dados por coordenadorias

In [ ]:
coordenadorias_dict = load_coordenadorias_dict(RAW_DIR)

coord_to_esec, esec_to_coord = build_coordenadoria_mappings(coordenadorias_dict)

In [ ]:
coordenadorias_dict.keys()

In [ ]:
print("Coordenadorias:", len(coord_to_esec))
print("ESECs:", len(esec_to_coord))

In [ ]:
from src.risk_processing import calcular_qtd_esecs_por_coordenadoria

esec_qtd_dict, total_esecs = calcular_qtd_esecs_por_coordenadoria(coord_to_esec)
print(total_esecs)

In [ ]:
from src.risk_processing import build_esec_municipio_mappings

esec_to_municipios, municipio_to_esec, total_municipios = (
    build_esec_municipio_mappings(coordenadorias_dict)
)

print(total_municipios)

In [ ]:
propriedades_df.columns.tolist()

In [ ]:
propriedades_por_esec = (
    propriedades_df.groupby("esec")
    .size()
    .to_dict()
)

In [ ]:
print(type(propriedades_df['codigo_do_municipio'].iloc[0]))
print(propriedades_df['codigo_do_municipio'].iloc[0])

primeira_chave = next(iter(municipio_to_esec))
print(type(primeira_chave))
print(primeira_chave)

In [ ]:
from src.risk_processing import normalizar_chaves_municipio

municipio_to_esec = normalizar_chaves_municipio(municipio_to_esec)

In [ ]:
from src.risk_processing import calcular_propriedades_por_esec

propriedades_por_esec = calcular_propriedades_por_esec(
    propriedades_df,
    municipio_to_esec
)

print(propriedades_por_esec)

In [ ]:
from src.risk_processing import construir_df_propriedades_por_esec

propriedades_por_esec_df = construir_df_propriedades_por_esec(propriedades_por_esec)

propriedades_por_esec_df.head()

In [ ]:
propriedades_por_esec_df['quantidade_propriedades'].sum()

In [ ]:
propriedades_por_esec_df[
    propriedades_por_esec_df["quantidade_propriedades"] == 0
]

In [ ]:
# testando quantas propriedades possui CAPELINHA
propriedades_por_esec_df[propriedades_por_esec_df['esec'] == 'Capelinha']

In [ ]:
from src.risk_processing import contar_propriedades_por_esec

propriedades_por_esec = contar_propriedades_por_esec(
    propriedades_df,
    esec_to_municipios
)

In [ ]:
from src.data_io import PROCESSED_DIR, salvar_propriedades_por_esec

path = salvar_propriedades_por_esec(
    propriedades_por_esec_df,
    PROCESSED_DIR
)

print(path)

## Riscos calculados

### Alimentação de suínos por lavagem

In [ ]:
from src.data_io import RAW_DIR, load_relatorio_suinos

relatorio_suinos_df = load_relatorio_suinos(RAW_DIR)

In [ ]:
relatorio_suinos_df.columns

In [ ]:
relatorio_suinos_df.head()

In [ ]:
len(relatorio_suinos_df)

#### Códigos de relevância para o tratamento dos dados relativos ao relatório dos suínos

* Códigos de relevância:         
    - CÓD. S7 – Fornecimento de resíduos alimentares (lavagem) aos suínos;  
    - CÓD. S1 – Proximidade de depósito de resíduos sólidos urbanos (lixão/aterro);   
    - CÓD. S4 – Proximidade de parada de ônibus proveniente de zona não livre;   
    - CÓD. 4 – Propriedades contíguas a aterros sanitários ou lixões;    
    - CÓD. 9 – Propriedades pertencentes ao mesmo proprietário em outros municípios, estados ou países.
* Colunas de relevância (ids das colunas):
    - 11 e 17 

In [ ]:
from src.risk_processing import extrair_codigos_suinos

relatorio_suinos_df[["Suinos_S", "Suinos_COD"]] = (
    relatorio_suinos_df.iloc[:, 11]
    .apply(lambda x: pd.Series(extrair_codigos_suinos(x)))
)

#### Criando novas colunas no dataframe relativas à exploração de suínos

Isso ajudará na melhor caracterização das áreas de risco (de acordo com os códigos acima).

In [ ]:
testes = [
    "CÓD S_123",
    "COD S 456",
    "COD 789",
    "CÓD -10",
]

for t in testes:
    print(t, "→", extrair_codigos_suinos(t))

In [ ]:
relatorio_suinos_df.head()

#### Criando novas colunas no dataframe relativas às propriedades

In [ ]:
# Criar novas colunas
relatorio_suinos_df[["Prop_S", "Prop_COD"]] = (
    relatorio_suinos_df.iloc[:, 17]
    .apply(lambda x: pd.Series(extrair_codigos_suinos(x)))
)

# Filtrar linhas com pelo menos um dos dois preenchido
relatorio_suinos_df = relatorio_suinos_df[
    relatorio_suinos_df["Prop_S"].notna() | relatorio_suinos_df["Prop_COD"].notna()
]

# Visualizar
print(relatorio_suinos_df[["Prop_S", "Prop_COD"]].head(20))

In [ ]:
relatorio_suinos_df.head()

In [ ]:
relatorio_suinos_df[["TIPO PRODUÇÃO TEC."]]

In [ ]:
from src.risk_processing import calcular_fatores_risco_suinos

fator_risco_suinos = calcular_fatores_risco_suinos(relatorio_suinos_df)

In [ ]:
# imprimir o dicionário (eliminar do notebook)
for cod, fatores in list(fator_risco_suinos.items())[:10]:
    print(f"Código: {cod}, Fatores: {fatores}, Quantidade: {len(fatores)}")

In [ ]:
relatorio_suinos_df["Suinos_S"].head(10)

In [ ]:
any("7" in x if isinstance(x, list) else False for x in relatorio_suinos_df["Suinos_S"])

In [ ]:
sum("BAIXA_BIOSSEGURIDADE" in fatores for fatores in fator_risco_suinos.values())

In [ ]:
from src.risk_processing import adicionar_risco_suinos

propriedades_df = adicionar_risco_suinos(
    propriedades_df,
    fator_risco_suinos,
)

In [ ]:
propriedades_df.head()

In [ ]:
# conta o número de propriedades em cada categoria de risco suínos
propriedades_df['risco_suinos'].value_counts()

In [ ]:
len(propriedades_df)

In [ ]:
from src.risk_processing import calcular_percentual_alto_risco_suinos

percentual = calcular_percentual_alto_risco_suinos(propriedades_df)

print(f"Percentual de propriedades em área de risco (suínos): {percentual:.2f}%")

### Proximidade de lixões

#### Configuração para a classificação das propriedades em relação à distância dos lixões

Utilizando, primeiramente, as informações do arquivo "relatorio_suinos.xlsx". Códigos de referência:  
* Códigos de relevância:         
    - CÓD. S1 – Proximidade de depósito de resíduos sólidos urbanos (lixão/aterro);   
    - CÓD. 4 – Propriedades contíguas a aterros sanitários ou lixões;    
* Colunas de relevância (ids das colunas):
    - 11 e 17 

In [ ]:
from src.data_io import RAW_DIR, load_lixoes

lixoes_df = load_lixoes(RAW_DIR)

len(lixoes_df)

In [ ]:
lixoes_df.head()

* Até 3 km: Propriedades em área de risco alto
* Entre 3 km e 10 km: Propriedades em área de atenção (risco médio)
* Acima de 10 km: Propriedades em área segura (risco baixo)

In [ ]:
propriedades_df["risco_lixoes"] = "baixo"

In [ ]:
from src.risk_processing import adicionar_distancia_lixao

propriedades_df = adicionar_distancia_lixao(
    propriedades_df,
    lixoes_df,
)

In [ ]:
from src.risk_processing import adicionar_risco_lixoes

propriedades_df = adicionar_risco_lixoes(propriedades_df)

In [ ]:
from src.risk_processing import adicionar_risco_lixoes_por_suinos_cod

propriedades_df = adicionar_risco_lixoes_por_suinos_cod(
    propriedades_df,
    relatorio_suinos_df
)

In [ ]:
from src.risk_processing import calcular_percentual_alto_risco_lixoes

percentual = calcular_percentual_alto_risco_lixoes(propriedades_df)

print(f"Percentual de propriedades em área de risco (lixões): {percentual:.2f}%")

### Proximidade a eventos

In [ ]:
from  src.data_io import RAW_DIR, load_eventos

relatorio_eventos_df = load_eventos(RAW_DIR)

In [ ]:
relatorio_eventos_df.head()

In [ ]:
from src.risk_processing import filtrar_eventos_alto_risco

eventos_considerados_df = filtrar_eventos_alto_risco(relatorio_eventos_df)

In [ ]:
propriedades_df["risco_eventos"] = "baixo"

In [ ]:
from src.risk_processing import adicionar_distancia_eventos

propriedades_df = adicionar_distancia_eventos(
    propriedades_df,
    eventos_considerados_df,
)

In [ ]:
from src.risk_processing import adicionar_risco_eventos

propriedades_df = adicionar_risco_eventos(propriedades_df)

In [ ]:
propriedades_df.head()

In [ ]:
propriedades_df["risco_eventos"].value_counts()

In [ ]:
propriedades_df.head()

### Densidade

- Cálculo com base na área do CAR  
- Regiões com população bovina acima do 3º quartil são consideradas de alta densidade.

In [ ]:
# verificar a existência de valores nulos no dataframe
propriedades_df["densidade_bovinos"].isnull().sum()

In [ ]:
propriedades_df['risco_densidade_bovinos'] = 'baixo'

In [ ]:
from src.risk_processing import adicionar_risco_densidade_bovinos

propriedades_df = adicionar_risco_densidade_bovinos(propriedades_df)

In [ ]:
propriedades_df["risco_densidade_bovinos"].value_counts()

### GTAs emitidas

- Análise do SIDAGRO com base nas GTAs.    
- Estabelecimentos com maior volume de entradas de bovinos.    
- Seleção de valores acima do 3º quartil.       
- Movimentações com mesma origem e destino na mesma data contam como um único movimento.

In [ ]:
from src.data_io import (
    RAW_DIR,
    load_gtas_entrada_origem_qualquer_MG_destino_prop_MG,
    load_gtas_entrada_origem_qualquer_OE_destino_prop_MG,
    load_gtas_saida_origem_prop_MG_destino_qualquer,
)

gtas_entrada_origem_qualquer_MG_destino_prop_MG_df = (
    load_gtas_entrada_origem_qualquer_MG_destino_prop_MG(RAW_DIR)
)

gtas_entrada_origem_qualquer_OE_destino_prop_MG_df = (
    load_gtas_entrada_origem_qualquer_OE_destino_prop_MG(RAW_DIR)
)

gtas_saida_origem_prop_MG_destino_qualquer_df = (
    load_gtas_saida_origem_prop_MG_destino_qualquer(RAW_DIR)
)

In [ ]:
# imprimir todos os possíveis valores únicos da coluna "estado_destino" do dataframe gtas_saida_origem_prop_MG_destino_qualquer_df
gtas_saida_origem_prop_MG_destino_qualquer_df["estado_destino"].unique()

In [ ]:
movimentacoes_por_propriedade_dict = {
    prop_id: None
    for prop_id in propriedades_df["codigo_da_propriedade"].unique()
}

In [ ]:
print(f"Antes: {len(gtas_entrada_origem_qualquer_OE_destino_prop_MG_df)}")
print(f"Antes: {len(gtas_entrada_origem_qualquer_MG_destino_prop_MG_df)}")

In [ ]:
from src.risk_processing import identificar_gtas_duplicadas

linhas_duplicadas, qtd_por_combinacao = identificar_gtas_duplicadas(
    gtas_entrada_origem_qualquer_OE_destino_prop_MG_df
)

In [ ]:
## Hardcoded de caso específico (remover na versão final)

filtro = (
    (linhas_duplicadas["dt_emissao_gta"] == "2024-10-01 00:00:00.0")
    & (linhas_duplicadas["cod_produtor_origem"] == 50567)
    & (linhas_duplicadas["cod_produtor_destino"] == 435231)
)

linhas_duplicadas[filtro]

In [ ]:
linhas_duplicadas2, qtd_por_combinacao2 = identificar_gtas_duplicadas(
    gtas_entrada_origem_qualquer_MG_destino_prop_MG_df
)

In [ ]:
## Hardcoded de caso específico (remover na versão final)

filtro = (
    (linhas_duplicadas2["dt_emissao_gta"] == "2025-09-14 00:00:00.0")
    & (linhas_duplicadas2["cod_produtor_origem"] == 81235)
    & (linhas_duplicadas2["cod_produtor_destino"] == 69443)
)

linhas_duplicadas2[filtro]

In [ ]:
from src.risk_processing import calcular_periodo_dias_gtas

total_dias = calcular_periodo_dias_gtas(
    gtas_entrada_origem_qualquer_MG_destino_prop_MG_df
)

print(total_dias)

In [ ]:
antes_filtrar_linhas_duplicadas_OE = len(
    gtas_entrada_origem_qualquer_OE_destino_prop_MG_df
)

antes_filtrar_linhas_duplicadas_MG = len(
    gtas_entrada_origem_qualquer_MG_destino_prop_MG_df
)

In [ ]:
from src.risk_processing import consolidar_gtas_mesma_origem_destino_data

gtas_entrada_origem_qualquer_OE_destino_prop_MG_df = (
    consolidar_gtas_mesma_origem_destino_data(
        gtas_entrada_origem_qualquer_OE_destino_prop_MG_df
    )
)

gtas_entrada_origem_qualquer_MG_destino_prop_MG_df = (
    consolidar_gtas_mesma_origem_destino_data(
        gtas_entrada_origem_qualquer_MG_destino_prop_MG_df
    )
)

In [ ]:
print(f"Depois de filtrar linhas duplicadas (Origem qualquer OE, destino prop MG): {len(gtas_entrada_origem_qualquer_OE_destino_prop_MG_df)}")
print(f"Depois de filtrar linhas duplicadas (Origem qualquer MG, destino prop MG): {len(gtas_entrada_origem_qualquer_MG_destino_prop_MG_df)}")

In [ ]:
print(f"Percentual de redução de linhas (Origem qualquer OE, destino prop MG): {((antes_filtrar_linhas_duplicadas_OE - len(gtas_entrada_origem_qualquer_OE_destino_prop_MG_df)) / antes_filtrar_linhas_duplicadas_OE * 100):.2f}%")
print(f"Percentual de redução de linhas (Origem qualquer MG, destino prop MG): {((antes_filtrar_linhas_duplicadas_MG - len(gtas_entrada_origem_qualquer_MG_destino_prop_MG_df)) / antes_filtrar_linhas_duplicadas_MG * 100):.2f}%")

In [ ]:
gtas_entrada_origem_qualquer_MG_destino_prop_MG_df.head(10)

In [ ]:
gtas_entrada_origem_qualquer_OE_destino_prop_MG_df.head(10)

In [ ]:
from src.risk_processing import calcular_movimentacoes_gtas_por_propriedade

movimentacoes_por_propriedade_dict = calcular_movimentacoes_gtas_por_propriedade(
    propriedades_df,
    gtas_entrada_origem_qualquer_OE_destino_prop_MG_df,
    gtas_entrada_origem_qualquer_MG_destino_prop_MG_df,
)

In [ ]:
from src.risk_processing import adicionar_metricas_gtas

propriedades_df = adicionar_metricas_gtas(
    propriedades_df,
    movimentacoes_por_propriedade_dict,
)

In [ ]:
propriedades_df.head()

In [ ]:
# quantidade de 'total_animais_gtas' igual a zero
(propriedades_df['total_animais_gtas'] == 0).sum()

In [ ]:
percentual_zero_animais_gtas = (
    (propriedades_df["total_animais_gtas"] == 0).sum()
    / len(propriedades_df)
)

print(
    f"Percentual de propriedades com total_animais_gtas igual a zero: "
    f"{percentual_zero_animais_gtas:.2%}"
)

In [ ]:
percentis_movimentacoes = propriedades_df[
    "total_movimentacoes_gtas"
].quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99])

percentis_movimentacoes

In [ ]:
propriedades_df['risco_gtas'] = 'baixo'

In [ ]:
from src.risk_processing import adicionar_risco_gtas

propriedades_df = adicionar_risco_gtas(propriedades_df)

In [ ]:
propriedades_df["risco_gtas"].value_counts()

## Calculando os riscos por coordenadoria

In [ ]:
from src.risk_processing import calcular_riscos_por_coordenadoria

propriedades_df = calcular_riscos_por_coordenadoria(
    propriedades_df,
    coord_to_esec,
)

In [ ]:
# estatistica descritiva da coluna 'densidade_bovinos'
propriedades_df['densidade_bovinos'].describe()

In [ ]:
from src.data_io import save_propriedades_risco_sanitario

save_propriedades_risco_sanitario(
    propriedades_df,
    PROCESSED_DIR,
)

In [ ]:
(propriedades_df[["risco_eventos"]] == 'alto').sum()

In [ ]:
propriedades_df.head(40)

In [ ]:
print("total_movimentacoes_gtas:")
print(propriedades_df['total_movimentacoes_gtas'].value_counts().sort_index())

print("\ntotal_animais_gtas:")
print(propriedades_df['total_animais_gtas'].value_counts().sort_index())